In [ ]:
import os
import sys
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go

# Point to project paths
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.config import RESULTS_DIR

print("Loading experiment results...")
df_dl = pd.read_csv(RESULTS_DIR / "dl_ablation_results.csv")
df_ml = pd.read_csv(RESULTS_DIR / "ml_baselines_results.csv")

combined_data = []

#  Process Deep Learning Data
dl_name_mapping = {
    "1. Multimodal (All Features)": "Deep ResNet (Multimodal, All Features)",
    "2. Multimodal (Magpie RFE)":   "Deep ResNet (Multimodal, Magpie RFE)",
    "3. Multimodal (Magpie RF)":    "Deep ResNet (Multimodal, Magpie RF)",
    "4. Tabular Only (All)":        "Deep ResNet (Tabular, All Features)",
    "5. Tabular Only (Magpie RFE)": "Deep ResNet (Tabular, Magpie RFE)",
    "6. Tabular Only (Magpie RF)":  "Deep ResNet (Tabular, Magpie RF)",
    "7. XRD Patterns Only":         "Deep ResNet (XRD, Raw Patterns)"
}

for _, row in df_dl.iterrows():
    raw_exp = row['Experiment']
    clean_name = dl_name_mapping.get(raw_exp, raw_exp)
    
    if "Multimodal" in raw_exp: cat = "Deep Learning (Multimodal)"
    elif "Tabular" in raw_exp: cat = "Deep Learning (Tabular Unimodal)"
    else: cat = "Deep Learning (XRD Unimodal)"
        
    combined_data.append({
        "Model Setup": clean_name, "Category": cat,
        "MAE (eV)": row['Test MAE (eV)'], "RMSE (eV)": row['Test RMSE (eV)'], "R2": row['Test R2']
    })

#  Process Traditional ML Dara
for _, row in df_ml.iterrows():
    # Clean up the reduction method strings
    red_method = row['Reduction Method'].replace("Magpie ", "").replace(" (24/30)", "")
    if red_method == "None (All Features)": red_method = "All Features"
    
    model_name = row['Model']
    clean_name = f"{model_name} (Tabular, {red_method})"
    
    if "Ridge" in model_name or "Lasso" in model_name: cat = "Traditional ML (Linear)"
    elif "Boost" in model_name or "Trees" in model_name: cat = "Traditional ML (Ensemble/Trees)"
    else: cat = "Traditional ML (MLP)"
        
    combined_data.append({
        "Model Setup": clean_name, "Category": cat,
        "MAE (eV)": row['MAE'], "RMSE (eV)": row['RMSE'], "R2": row['R2']
    })

# Sort by MAE
df_all = pd.DataFrame(combined_data).sort_values(by="MAE (eV)", ascending=True)
print(f"Merged {len(df_all)} total model evaluations.")

Loading experiment results...
Merged 23 total model evaluations.


In [2]:
color_map = {
    "Deep Learning (Multimodal)": "#2ca02c",      
    "Traditional ML (Ensemble/Trees)": "#1f77b4", 
    "Traditional ML (MLP)": "#9467bd",            
    "Deep Learning (Tabular Unimodal)": "#ff7f0e",
    "Deep Learning (XRD Unimodal)": "#17becf",    
    "Traditional ML (Linear)": "#7f7f7f"          
}

fig = go.Figure()

# Add the MAE Bars 
for cat, color in color_map.items():
    df_cat = df_all[df_all["Category"] == cat]
    if not df_cat.empty:
        fig.add_trace(go.Bar(
            x=df_cat["Model Setup"],  
            y=df_cat["MAE (eV)"],     
            name=cat,
            marker_color=color,
            hovertemplate="<b>%{x}</b><br>MAE: %{y:.4f} eV<extra></extra>"
        ))

# Add the R2 Connected Trendline 
fig.add_trace(go.Scatter(
    x=df_all["Model Setup"], 
    y=df_all["R2"],          
    mode='lines+markers', 
    name='R² Score',
    yaxis='y2',              
    line=dict(color='#ff6666', width=2), # Light Red
    marker=dict(symbol='circle', size=8, color='#ff6666'), 
    hovertemplate="<b>%{x}</b><br>R²: %{y:.4f}<extra></extra>"
))

# Format Layout and Dual Axes
fig.update_layout(
    title="Comparison Between Different Models and Feature Selection",
    height=850, width=1400, 
    plot_bgcolor="white",
    hoverlabel=dict(bgcolor="white", font_size=14, font_family="Arial"),
    barmode='group',
    legend=dict(title="Model Category", x=1.08, y=1, xanchor="left", yanchor="top"),
    
    xaxis=dict(
        title="",
        categoryorder='array',
        categoryarray=df_all["Model Setup"].tolist(),
        tickangle=-45, 
        showgrid=False
    ),
    
    yaxis=dict(
        title="<b>Test MAE (eV)</b>",
        showgrid=True, gridwidth=1, gridcolor='LightGray',
        side='left'
    ),
    
    yaxis2=dict(
        title="<b>Test R²</b>",
        overlaying='y', 
        side='right',
        showgrid=False,
        range=[0, max(df_all["R2"]) * 1.1] 
    ),
    margin=dict(b=200) 
)

# Save to the results folder
out_path = RESULTS_DIR / "interactive_benchmark.html"
fig.write_html(out_path)
print(f"Saved interactive chart to: {out_path}")

fig.show()

Saved interactive chart to: /home/users/yunchilo/Desktop/Bandgap-Predictor/results/interactive_benchmark.html
